In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

In [ ]:
caminho_pdf = ("documents/inteligencia_artificial.pdf")

loader = PyPDFLoader(str(caminho_pdf))

documentos = loader.load()

print(f"Total de páginas carregadas: {len(documentos)}")
print(documentos[0].page_content[:500])

In [ ]:
divisor = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

trechos = divisor.split_documents(documentos)

print(f"Quantidade de trechos: {len(trechos)}")
print(trechos[0].page_content)

In [ ]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vetor_teste = embeddings.embed_query(
    "O que é inteligência artificial?"
)

print(f"Dimensões do vetor: {len(vetor_teste)}")
print(vetor_teste[:10])

In [ ]:
banco_vetorial = FAISS.from_documents(
    documents=trechos,
    embedding=embeddings
)

print("Banco vetorial criado com sucesso!")

In [ ]:
pergunta = "Do que se trata o documento?"

resultados = banco_vetorial.similarity_search(
    pergunta,
    k=3
)

for i, documento in enumerate(resultados, start=1):
    print(f"\n--- Resultado {i} ---")
    print(documento.page_content[:500])

In [ ]:
modelo = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

recuperador = banco_vetorial.as_retriever(
    search_kwargs={"k": 3}
)

qa = RetrievalQA.from_chain_type(
    llm=modelo,
    chain_type="stuff",
    retriever=recuperador,
)

print("RetrievalQA configurado!")

In [ ]:
pergunta = input("Digite sua pergunta sobre o documento: ")

resultado = qa.invoke({
    "query": pergunta
})

print("\nRESPOSTA:")
print(resultado["result"])